<a href="https://colab.research.google.com/github/prabhnoor14/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [4]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [5]:
# TODO (hint: .str.lower().str.contains('pop'))
mask_pop = cat['product_name'].str.lower().str.contains('pop', na=False)
print("Rows containing 'pop':", mask_pop.sum())
cat[mask_pop]

Rows containing 'pop': 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


Explaination: I lowercased the names and searched for “pop”, which found 4 rows, and all of them are strawberry Pop-Tarts, so nothing unrelated slipped in. The same product shows up under 4 different SKUs, with the category written three ways (“Breakfast Foods”, “Breakfast”, “breakfast”) and package size written four ways.One row is a 6 pack instead of a 4 pack, so it might not be the same product.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [6]:
# TODO
mask_pt = cat['product_name'].str.contains(r'pop[\s\-]?tart', case=False, na=False)
is_straw = mask_pt & cat['product_name'].str.contains('straw', case=False, na=False)

print("Strawberry Pop-Tart variants:", is_straw.sum())
display(cat.loc[is_straw, ['sku', 'product_name', 'package_size']])

print("Pop-Tart rows that are NOT strawberry:", (mask_pt & ~is_straw).sum())
display(cat.loc[mask_pt & ~is_straw, ['sku', 'product_name', 'package_size']])

Strawberry Pop-Tart variants: 4


,sku,product_name,package_size
0,PT-12,pop-tarts strawberry,4-pack
1,SKU#459812,Strawberry Pop-Tart,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,4 pack


Pop-Tart rows that are NOT strawberry: 0


,sku,product_name,package_size


Explaination: I narrowed the search to names matching “pop-tart” and then to ones containing “straw”, which kept all 4 rows. All four SKUs mean strawberry Pop-Tart, and no Pop-Tart rows are a different flavor. The only possible exception is POPTART-STRAW, which is a 6 pack while the others are 4 packs.

**Pop-Tart rows that are a different product:** : none by flavor. POPTART-STRAW (6 pack) might count as separate because of its pack size, i am not assuming it’s the same.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [7]:
# TODO
CANONICAL = 'PT-12'

sku_map = {
    'SKU#459812':    CANONICAL,
    'PT_STRAWBERRY': CANONICAL,
    # 'POPTART-STRAW': CANONICAL,    6 pack: left out on purpose
}

cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

print("Unique SKUs before:", cat['sku'].nunique())
print("Unique SKUs after: ", cat['canonical_sku'].nunique())
display(cat.loc[is_straw, ['sku', 'canonical_sku', 'product_name', 'package_size']])

Unique SKUs before: 45
Unique SKUs after:  43


,sku,canonical_sku,product_name,package_size
0,PT-12,PT-12,pop-tarts strawberry,4-pack
1,SKU#459812,PT-12,Strawberry Pop-Tart,4pk
2,POPTART-STRAW,POPTART-STRAW,Pop-Tarts Strawberry,6 pack
3,PT_STRAWBERRY,PT-12,pop-tarts strawberry,4 pack


explaination: I mapped the two duplicate 4-pack SKUs to PT-12 and used .fillna(cat['sku']) so every SKU not in the map kept its original value instead of turning into NaN. Unique SKUs dropped from 45 to 43, nothing else in the catalog changed. I left POPTART-STRAW separate because it’s a 6 pack, and a different pack size could be its own product.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [9]:
# TODO
print("BEFORE:")
print(cat['category'].value_counts(dropna=False), "\n")

cat['category_clean'] = cat['category'].str.strip().str.title()

print("AFTER strip + title-case:")
print(cat['category_clean'].value_counts(dropna=False), "\n")

category_map = {
    'Breakfast Foods':     'Breakfast',
    'Emergency Supplies':  'Emergency',
    'Alcoholic Beverages': 'Alcohol',
    'Groceries':           'Grocery',
    'Baby Products':       'Baby',
}
cat['category_clean'] = cat['category_clean'].replace(category_map)

print("AFTER merging synonyms:")
print(cat['category_clean'].value_counts(dropna=False))

BEFORE:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64 

AFTER strip + title-case:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health        

explaination: I cleaned up the category names by fixing capitalization and merging similar labels, which reduced the categories. I left Breakfast and Grocery separate since I wasn't sure if they should be combined.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [10]:
# TODO
import re

def name_key(name):
    s = str(name).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)    # punctuation
    words = [w[:-1] if w.endswith('s') and len(w) > 3 else w for w in s.split()]
    return ' '.join(sorted(words))         # word order doesn't matter

cat['name_key'] = cat['product_name'].apply(name_key)

sku_counts = cat.groupby('name_key')['sku'].nunique()
fragmented = sku_counts[sku_counts > 1].sort_values(ascending=False)
print("Name groups with more than one SKU:", len(fragmented))

display(cat[cat['name_key'].isin(fragmented.index)]
        .sort_values('name_key')
        [['name_key', 'sku', 'product_name', 'category_clean', 'package_size']])

Name groups with more than one SKU: 14


,name_key,sku,product_name,category_clean,package_size
12,24 beer light pack,BEER-24,Light Beer (24-pack),Alcohol,bundle
13,24 beer light pack,LTBEER24,Light Beer (24-pack),Alcohol,bundle
8,4 aa batterie pack,BAT-AA4,AA Batteries 4-Pack,Emergency,4pk
9,4 aa batterie pack,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,24 pack
30,aid first kit,FIRSTAID-KIT,First Aid Kit,Health,4 pack
29,aid first kit,FAID-01,First-Aid Kit,Health,4 pack
14,board game monopoly,BG-MONO,Monopoly Board Game,Entertainment,1pk
16,board game monopoly,GAME-MONO,Board Game - Monopoly,Entertainment,1pk
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,Entertainment,24-pack
24,bread white,WHITEBREAD,WHITE BREAD,Grocery,24 pack


explaination: I found that many products have multiple SKUs, so duplicates seem to be common in this catalog. Some product details are also inconsistent, so it's hard to tell which products should be merged. This also makes me reconsider keeping the Pop-Tart 6-pack separate.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [11]:
# TODO
n_before = len(cat)

out = cat.assign(canonical_sku=cat['sku'].map(sku_map).fillna(cat['sku']))

# 1. No nulls in the canonical column
assert out['canonical_sku'].notna().all(), "Mapping produced null SKUs"

# 2. Same number of rows
assert len(out) == n_before, "Row count changed"

# 3. SKUs not in the map are unchanged
unmapped = ~out['sku'].isin(sku_map.keys())
assert (out.loc[unmapped, 'canonical_sku'] == out.loc[unmapped, 'sku']).all(), "Unmapped SKUs changed"

print(f"Mapping is safe: {len(out)} rows, 0 nulls, "
      f"unique SKUs {out['sku'].nunique()} -> {out['canonical_sku'].nunique()}")

Mapping is safe: 45 rows, 0 nulls, unique SKUs 45 -> 43


explaination: I checked the mapping to make sure it didn't change the row count, create any nulls, or change SKUs that weren't mapped. All checks passed, and the number of unique SKUs went from 45 to 43 after merging two SKUs.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a)  Monopoly has three SKUs, and AA batteries have two. I would group products with the same name and map them to one SKU, like I did for Pop-Tarts.
b) It becomes a business decision when I'm not sure if two products are actually the same. For example, I'm not sure if the Pop-Tart 6-pack should be merged or kept separate. I'd ask the category manager to confirm whether they are sold as different products.